# 05 - RFM Customer Segmentation
## Global Superstore Dataset

**الهدف:** إضافة بعد تحليلي جديد على مستوى العميل (مش المنتج أو السوق بس) — مين أهم عملاء الشركة فعليًا،
ومين على وشك يسيبها؟

**RFM = Recency, Frequency, Monetary**
- **Recency:** من كام يوم آخر مرة العميل اشترى فيها؟ (كل ما أقل كل ما أحسن)
- **Frequency:** كام طلب منفصل عمله العميل؟
- **Monetary:** إجمالي قيمة مشترياته؟

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_cleaning import load_and_clean_data
from rfm_analysis import compute_rfm, score_rfm, segment_customers, segment_summary

sns.set_theme(style='whitegrid')
df = load_and_clean_data('../data/raw/superstore_dataset2011-2015.csv')
df.head()

## 1. Build the RFM Table

استخدمنا `Order ID` مش `Sales` مباشرة لحساب الـ Frequency (نفس درس AOV من قبل —
مينفعش نعد الأسطر، لازم نعد الطلبات الفريدة).

In [ ]:
rfm = compute_rfm(df)
print(rfm.describe())
rfm.head()

## 2. Score Each Dimension (1-5 Quintiles) and Assign Segments

In [ ]:
rfm_scored = segment_customers(df)
print(rfm_scored['RFM_Segment'].value_counts())
rfm_scored.head()

## 3. Segment Summary

مين بيجيب فلوس فعليًا، ومين مجرد عدد كبير من غير قيمة حقيقية؟

In [ ]:
summary = segment_summary(rfm_scored)
print(summary)

### 🎯 Insight: تركز الإيراد في شريحة صغيرة من العملاء

Champions (~21.8% من العملاء) بيجيبوا ~43.3% من إجمالي الإيراد — ده مبدأ Pareto (80/20) ظاهر بوضوح.
في المقابل، Lost (أكبر شريحة عدديًا: 367 عميل) بتجيب 3.6% بس من الإيراد.

## 4. Visualizations

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(figsize=(9,5))
sns.countplot(data=rfm_scored, y='RFM_Segment', order=order, hue='RFM_Segment', legend=False, palette='viridis', ax=ax)
ax.set_title('Customer Count by RFM Segment')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=summary.reset_index(), y='RFM_Segment', x='Pct_of_Revenue', order=order, hue='RFM_Segment', legend=False, palette='magma', ax=ax)
ax.set_title('% of Total Revenue by RFM Segment')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
sns.scatterplot(data=rfm_scored, x='Recency', y='Frequency', hue='RFM_Segment',
                size='Monetary', sizes=(20,300), alpha=0.7, palette='tab10', ax=ax)
ax.set_title('Customer Segments: Recency vs Frequency (bubble size = Monetary)')
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.show()

## 5. Interactive Version (Plotly)

**ملحوظة:** الكود ده جاهز يشتغل عندك في جهازك (`pip install plotly`)، لأن بيئة التنفيذ هنا معندهاش اتصال إنترنت لتثبيت مكتبات جديدة.
النتيجة هتكون رسمة تفاعلية تقدر تكبّر فيها وتشوف كل عميل لما تمرّر الماوس عليه.

In [ ]:
import plotly.express as px

fig = px.scatter(
    rfm_scored.reset_index(), x='Recency', y='Frequency',
    color='RFM_Segment', size='Monetary', hover_name='Customer ID',
    title='Interactive RFM Segmentation: Recency vs Frequency',
    size_max=40, opacity=0.7
)
fig.update_layout(width=950, height=600)
fig.show()

In [ ]:
import plotly.express as px

fig2 = px.treemap(
    summary.reset_index(), path=['RFM_Segment'], values='Total_Monetary',
    color='Avg_Monetary', color_continuous_scale='Blues',
    title='Revenue Contribution by Segment (Treemap)'
)
fig2.update_layout(width=800, height=500)
fig2.show()

## 6. Business Recommendation Based on RFM

| Segment | ملاحظة | التوصية |
|---|---|---|
| **Champions** (347 عميل، 43.3% إيراد) | أهم شريحة بفارق كبير | برنامج ولاء VIP، وصول مبكر لعروض جديدة |
| **At Risk (High Value)** (77 عميل، 9.1% إيراد) | كانوا بيصرفوا كتير لكن مبيشتروش من فترة | حملة Win-back مخصصة قبل ما يتحولوا لـ Lost |
| **Loyal Customers** (355 عميل، 31.9% إيراد) | مستقرين، أساس الإيراد المتكرر | الحفاظ على تجربة الخدمة، Upselling تدريجي |
| **Lost** (367 عميل، 3.6% إيراد بس) | أكبر عدد لكن أقل قيمة | لا يستحق استثمار تسويقي كبير — أولوية منخفضة |

### التحقق من فرضية: هل At Risk ابتعدوا بسبب الخصم؟

فحصنا هل شريحة **At Risk (High Value)** كانت بتاخد خصومات أعلى من غيرها (ربطًا بـ Insight #1 عن الخصم) — والنتيجة نفت الفرضية:


In [ ]:
merged = df.merge(rfm_scored[['RFM_Segment']], left_on='Customer ID', right_index=True)
seg_discount = merged.groupby('RFM_Segment')['Discount'].mean().sort_values(ascending=False)
print(seg_discount.round(3))

**النتيجة:** At Risk (High Value) عندها **أقل** متوسط خصم (13.3%) بين كل الشرائح — مش الأعلى. يبقى **الفرضية اتفحصت واتفنّدت بأمانة**، زي ما حصل مع Ship Mode قبل كده. سبب ابتعادهم مش الخصم، ولازم بحث إضافي (رضا عن المنتج؟ منافس؟ تغيّر في احتياجهم؟) — الداتا الحالية مقدرش تجاوب على السؤال ده لوحدها، وده حد صادق للتحليل.